# Phase 9: Learnable `(a,b)` OSP Bridge

Goal: build a constrained learnable bridge back to the original OSP generator family.

Pattern family:
`G = mod(I * a + J * b, NF) + 1`

This notebook does not replace the full learned MSFA model.
It is the OSP-faithful bridge experiment.


# Learnable MSFA Research Track

This notebook is part of the 10-day publication-oriented extension track.

## Run discipline
- Keep one experiment purpose per notebook.
- Save metrics and checkpoints to the configured project root.
- Do not silently change data splits or loss definitions across notebooks.
- When a result becomes final, export both numeric artifacts and a figure/table asset.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
except Exception as exc:
    print("Drive mount skipped:", exc)


In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path("/content/drive/MyDrive/Msa-Osp")
PATCH_PATH = PROJECT_ROOT / "dataset_patches.npz"
HISTORY_PATH = PROJECT_ROOT / "learnable_ab_history.csv"
CKPT_PATH = PROJECT_ROOT / "learnable_ab_best.pth"
FIG_PATH = PROJECT_ROOT / "learnable_ab_pattern.png"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
LR = 2e-4
BAND_COUNT = 16
TILE_SIZE = 16
A_MAX = BAND_COUNT / 2
B_MAX = BAND_COUNT / 2
EPOCHS = 80
SHARPNESS_START = 8.0
SHARPNESS_END = 20.0
LAMBDA_OSP = 1e-3
LAMBDA_UNIFORMITY = 5e-3
LAMBDA_ENTROPY = 1e-3
D_MIN_3D = 0.55
Z_WEIGHT = 1.25

print("Device:", DEVICE)


In [ ]:
data = np.load(PATCH_PATH)
train_target = data["train"]
val_target = data["val"]

class CubeDataset(Dataset):
    def __init__(self, cubes):
        self.cubes = cubes

    def __len__(self):
        return len(self.cubes)

    def __getitem__(self, idx):
        return torch.from_numpy(self.cubes[idx]).permute(2, 0, 1).float()

train_loader = DataLoader(CubeDataset(train_target), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(CubeDataset(val_target), batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet2D(nn.Module):
    def __init__(self, in_ch=16, out_ch=16, base=64):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 2, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.final = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final(d1)

class LearnableABMask(nn.Module):
    def __init__(self, bands=BAND_COUNT, tile_size=TILE_SIZE):
        super().__init__()
        self.bands = bands
        self.tile_size = tile_size
        self.raw_a = nn.Parameter(torch.tensor(0.0))
        self.raw_b = nn.Parameter(torch.tensor(0.0))

        i = torch.arange(1, tile_size + 1, dtype=torch.float32)
        j = torch.arange(1, tile_size + 1, dtype=torch.float32)
        I, J = torch.meshgrid(i, j, indexing="ij")
        self.register_buffer("I", I)
        self.register_buffer("J", J)
        self.register_buffer("band_ids", torch.arange(1, bands + 1, dtype=torch.float32).view(1, 1, bands))

    def continuous_ab(self):
        a = 1.0 + (A_MAX - 1.0) * torch.sigmoid(self.raw_a)
        b = 1.0 + (B_MAX - 1.0) * torch.sigmoid(self.raw_b)
        return a, b

    def soft_tile(self, sharpness=SHARPNESS_START):
        a, b = self.continuous_ab()
        value = torch.remainder(self.I * a + self.J * b - 1.0, float(self.bands)) + 1.0
        dist = torch.abs(value.unsqueeze(-1) - self.band_ids)
        circular = torch.minimum(dist, float(self.bands) - dist)
        logits = -sharpness * circular
        weights = torch.softmax(logits, dim=-1)
        return weights.permute(2, 0, 1)

    def hard_tile(self, sharpness=SHARPNESS_END):
        return self.soft_tile(sharpness=sharpness).argmax(dim=0) + 1

    def forward(self, x, sharpness=SHARPNESS_START):
        _, _, h, w = x.shape
        tile = self.soft_tile(sharpness=sharpness)
        tile_full = tile.repeat(1, h // self.tile_size, w // self.tile_size)
        return x * tile_full.unsqueeze(0)

def compute_psnr(pred, target, eps=1e-8):
    mse = torch.mean((pred - target) ** 2)
    return 20 * torch.log10(1.0 / torch.sqrt(mse + eps))

def spectral_angle_mapper(pred, target, eps=1e-8):
    dot = torch.sum(pred * target, dim=1)
    pred_norm = torch.norm(pred, dim=1)
    target_norm = torch.norm(target, dim=1)
    cos_theta = torch.clamp(dot / (pred_norm * target_norm + eps), -1 + eps, 1 - eps)
    return torch.rad2deg(torch.acos(cos_theta)).mean()

spatial_coords = torch.stack(
    torch.meshgrid(
        torch.linspace(0.0, 1.0, TILE_SIZE),
        torch.linspace(0.0, 1.0, TILE_SIZE),
        indexing="ij",
    ),
    dim=-1,
).reshape(-1, 2)

wavelength_coords = torch.linspace(0.0, 1.0, BAND_COUNT)

def bridge_points(mask_model, sharpness=SHARPNESS_START):
    soft_tile = mask_model.soft_tile(sharpness=sharpness)
    z = (soft_tile * wavelength_coords.to(soft_tile.device).view(BAND_COUNT, 1, 1)).sum(dim=0).reshape(-1, 1)
    xy = spatial_coords.to(soft_tile.device)
    return torch.cat([xy, Z_WEIGHT * z], dim=1)

def bridge_osp_loss(mask_model, d_min=D_MIN_3D, sharpness=SHARPNESS_START):
    points = bridge_points(mask_model, sharpness=sharpness)
    d = torch.cdist(points, points, p=2)
    mask = torch.triu(torch.ones_like(d), diagonal=1) > 0
    distances = d[mask]
    return torch.relu(d_min - distances).mean()

def bridge_distance_stats(mask_model, sharpness=SHARPNESS_END):
    points = bridge_points(mask_model, sharpness=sharpness)
    d = torch.cdist(points, points, p=2)
    mask = torch.triu(torch.ones_like(d), diagonal=1) > 0
    distances = d[mask]
    return {
        "min_3d_distance": distances.min().item(),
        "mean_3d_distance": distances.mean().item(),
    }

def uniformity_loss(mask_model, sharpness=SHARPNESS_START):
    soft_tile = mask_model.soft_tile(sharpness=sharpness)
    usage = soft_tile.mean(dim=(1, 2))
    target = torch.full_like(usage, 1.0 / BAND_COUNT)
    return torch.mean((usage - target) ** 2)

def entropy_loss(mask_model, sharpness=SHARPNESS_START, eps=1e-8):
    soft_tile = mask_model.soft_tile(sharpness=sharpness)
    entropy = -(soft_tile * torch.log(soft_tile + eps)).sum(dim=0)
    return entropy.mean()


In [ ]:
model = UNet2D().to(DEVICE)
mask_model = LearnableABMask().to(DEVICE)
optimizer = torch.optim.Adam(
    [
        {"params": model.parameters(), "lr": LR},
        {"params": mask_model.parameters(), "lr": LR},
    ]
)
criterion = nn.L1Loss()
best_psnr = -float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    mask_model.train()
    train_loss = 0.0
    train_psnr = 0.0
    sharpness = SHARPNESS_START + (SHARPNESS_END - SHARPNESS_START) * (epoch - 1) / max(EPOCHS - 1, 1)

    for x in train_loader:
        x = x.to(DEVICE)
        sensed = mask_model(x, sharpness=sharpness)
        pred = model(sensed)
        recon_loss = criterion(pred, x)
        osp_loss = bridge_osp_loss(mask_model, sharpness=sharpness)
        uniform_loss = uniformity_loss(mask_model, sharpness=sharpness)
        discrete_loss = entropy_loss(mask_model, sharpness=sharpness)
        loss = (
            recon_loss
            + LAMBDA_OSP * osp_loss
            + LAMBDA_UNIFORMITY * uniform_loss
            + LAMBDA_ENTROPY * discrete_loss
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_psnr += compute_psnr(pred, x).item()

    model.eval()
    mask_model.eval()
    val_psnr = 0.0
    val_sam = 0.0
    with torch.no_grad():
        for x in val_loader:
            x = x.to(DEVICE)
            pred = model(mask_model(x, sharpness=sharpness))
            val_psnr += compute_psnr(pred, x).item()
            val_sam += spectral_angle_mapper(pred, x).item()

    train_loss /= len(train_loader)
    train_psnr /= len(train_loader)
    val_psnr /= len(val_loader)
    val_sam /= len(val_loader)
    a, b = mask_model.continuous_ab()
    stats = bridge_distance_stats(mask_model, sharpness=sharpness)
    osp_value = bridge_osp_loss(mask_model, sharpness=sharpness).item()
    uniform_value = uniformity_loss(mask_model, sharpness=sharpness).item()
    discrete_value = entropy_loss(mask_model, sharpness=sharpness).item()
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_psnr": train_psnr,
            "val_psnr": val_psnr,
            "val_sam_deg": val_sam,
            "a_cont": a.item(),
            "b_cont": b.item(),
            "sharpness": sharpness,
            "bridge_osp_loss": osp_value,
            "uniformity_loss": uniform_value,
            "entropy_loss": discrete_value,
            "min_3d_distance": stats["min_3d_distance"],
            "mean_3d_distance": stats["mean_3d_distance"],
        }
    )
    print(
        f"Epoch {epoch:02d} | train PSNR {train_psnr:.2f} dB | "
        f"val PSNR {val_psnr:.2f} dB | val SAM {val_sam:.2f} deg | "
        f"a {a.item():.3f} | b {b.item():.3f} | "
        f"osp {osp_value:.4f} | uni {uniform_value:.4f} | ent {discrete_value:.4f} | "
        f"min 3D d {stats['min_3d_distance']:.3f} | sharp {sharpness:.1f}"
    )

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "mask_model": mask_model.state_dict(),
                "best_val_psnr": best_psnr,
                "a_cont": a.item(),
                "b_cont": b.item(),
                "sharpness": sharpness,
                "lambda_osp": LAMBDA_OSP,
                "lambda_uniformity": LAMBDA_UNIFORMITY,
                "lambda_entropy": LAMBDA_ENTROPY,
                "d_min_3d": D_MIN_3D,
                "z_weight": Z_WEIGHT,
                "distance_stats": stats,
                "hard_tile": mask_model.hard_tile(sharpness=sharpness).detach().cpu().numpy(),
            },
            CKPT_PATH,
        )

HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(HISTORY_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    writer.writeheader()
    writer.writerows(history)


In [ ]:
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
mask_model.load_state_dict(checkpoint["mask_model"])
final_sharpness = checkpoint.get("sharpness", SHARPNESS_END)
hard_tile = mask_model.hard_tile(sharpness=final_sharpness).detach().cpu().numpy()
soft_tile = mask_model.soft_tile(sharpness=final_sharpness).detach().cpu().numpy()

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(hard_tile, cmap="tab20")
plt.title(f"Learnable (a,b) hard tile\na={checkpoint['a_cont']:.3f}, b={checkpoint['b_cont']:.3f}")
plt.colorbar()

plt.subplot(1, 2, 2)
for i in range(soft_tile.shape[1]):
    for j in range(soft_tile.shape[2]):
        plt.plot(soft_tile[:, i, j], alpha=0.7)
plt.title("Soft assignments induced by learnable (a,b)")
plt.xlabel("Band")
plt.ylabel("Weight")
plt.tight_layout()
FIG_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_PATH, dpi=200, bbox_inches="tight")
plt.show()

print("Best checkpoint sharpness:", final_sharpness)
print("Best checkpoint distance stats:", checkpoint.get("distance_stats", {}))


## How to use this notebook

Use it as the bridge experiment:
- `Fixed OSP`
- `Learnable (a,b)` OSP-family bridge
- `Learned MSFA + 3D SP`

If this bridge underperforms the full learned model, that strengthens the argument for moving beyond the original constrained OSP family.
